# Stage 03 — HELOC Data Preprocessing

This notebook freezes the modelling cohort, holdout partition and training-fitted preprocessing pipeline. It follows the approved Stage 02 decisions without further feature selection or model development.

## 1. Frozen inputs and integrity

Only the immutable local snapshot and persisted Stage 01–02 artifacts are loaded. The snapshot hash is checked before any cohort construction.

In [1]:
from pathlib import Path
import hashlib
import json

import joblib
import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.utils.validation import check_is_fitted

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 180)

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "heloc": PROJECT_ROOT = PROJECT_ROOT.parents[2]
elif PROJECT_ROOT.name == "ml": PROJECT_ROOT = PROJECT_ROOT.parents[1]
elif PROJECT_ROOT.name == "backend": PROJECT_ROOT = PROJECT_ROOT.parent

RAW_PATH = PROJECT_ROOT / "backend" / "data" / "heloc" / "raw" / "heloc_raw.csv"
ARTIFACT_DIR = PROJECT_ROOT / "backend" / "artifacts" / "heloc"
EXPECTED_HASH = "786b7e1885cf508ef66a968aae65a72bfde198d06d7666ac582958705b092570"
TARGET = "is_at_risk"
SPECIAL_CODES = (-7, -8, -9)
TEST_SIZE = 0.20
RANDOM_STATE = 42

required_inputs = [
    "selected_features.json", "stage03_recommendation.json", "special_value_feature_audit.csv",
    "duplicate_structure_audit.csv", "feature_semantics_audit.csv", "xai_actionability_audit.csv",
    "dataset_provenance.json",
]
for name in required_inputs:
    if not (ARTIFACT_DIR / name).exists(): raise FileNotFoundError(name)

actual_hash = hashlib.sha256(RAW_PATH.read_bytes()).hexdigest()
assert actual_hash == EXPECTED_HASH
raw_df = pd.read_csv(RAW_PATH)
with open(ARTIFACT_DIR / "selected_features.json", encoding="utf-8") as handle: selected_payload = json.load(handle)
with open(ARTIFACT_DIR / "stage03_recommendation.json", encoding="utf-8") as handle: stage03_decision = json.load(handle)
with open(ARTIFACT_DIR / "dataset_provenance.json", encoding="utf-8") as handle: provenance = json.load(handle)
special_stage02 = pd.read_csv(ARTIFACT_DIR / "special_value_feature_audit.csv")
duplicate_stage02 = pd.read_csv(ARTIFACT_DIR / "duplicate_structure_audit.csv")
semantics_stage02 = pd.read_csv(ARTIFACT_DIR / "feature_semantics_audit.csv")
actionability_stage02 = pd.read_csv(ARTIFACT_DIR / "xai_actionability_audit.csv")

selected_features = selected_payload["selected_features"]
excluded_features = selected_payload["excluded_features"]
assert raw_df.shape == (10_459, 24)
assert provenance["target_column"] == TARGET and set(raw_df[TARGET]) == {0, 1}
assert len(selected_features) == 22 and "estimate_of_risk" not in selected_features
print("Raw SHA-256 verified:", actual_hash)
print("Selected predictors:", len(selected_features))
print("Target: 0 = not at risk; 1 = at risk")

Raw SHA-256 verified: 786b7e1885cf508ef66a968aae65a72bfde198d06d7666ac582958705b092570
Selected predictors: 22
Target: 0 = not at risk; 1 = at risk


## 2. Modelling cohort row policy

All 588 rows with `-9` across the original 23 predictors are excluded from modelling. Remaining exact duplicates are rechecked, with only deterministic excess copies excluded.

In [2]:
original_predictors = [column for column in raw_df.columns if column != TARGET]
all_minus9_mask = raw_df[original_predictors].eq(-9).all(axis=1)
excluded_all_minus9_indices = raw_df.index[all_minus9_mask].astype(int)
all_minus9_rows = raw_df.loc[all_minus9_mask]
assert len(excluded_all_minus9_indices) == 588
assert all_minus9_rows[TARGET].value_counts().sort_index().to_dict() == {0: 265, 1: 323}

after_all_minus9 = raw_df.loc[~all_minus9_mask].copy()
remaining_duplicate_mask = after_all_minus9.duplicated(keep="first")
excluded_duplicate_indices = after_all_minus9.index[remaining_duplicate_mask].astype(int)
assert len(excluded_duplicate_indices) == 1

modelling_df = after_all_minus9.loc[~remaining_duplicate_mask].copy()
assert len(modelling_df) == 9_870

pd.DataFrame({"row_index": excluded_all_minus9_indices}).to_csv(ARTIFACT_DIR / "excluded_all_minus9_indices.csv", index=False)
pd.DataFrame({"row_index": excluded_duplicate_indices}).to_csv(ARTIFACT_DIR / "excluded_duplicate_indices.csv", index=False)

before_counts = raw_df[TARGET].value_counts().sort_index()
after_minus9_counts = after_all_minus9[TARGET].value_counts().sort_index()
final_counts = modelling_df[TARGET].value_counts().sort_index()
modelling_cohort_audit = pd.DataFrame([{
    "raw_rows": len(raw_df), "all_minus9_rows_excluded": len(excluded_all_minus9_indices),
    "remaining_rows_after_all_minus9_exclusion": len(after_all_minus9),
    "duplicate_excess_excluded": len(excluded_duplicate_indices), "final_modelling_rows": len(modelling_df),
    "raw_class_0_count": int(before_counts[0]), "raw_class_1_count": int(before_counts[1]),
    "raw_class_0_percentage": 100 * before_counts[0] / len(raw_df), "raw_class_1_percentage": 100 * before_counts[1] / len(raw_df),
    "after_all_minus9_class_0_count": int(after_minus9_counts[0]), "after_all_minus9_class_1_count": int(after_minus9_counts[1]),
    "after_all_minus9_class_0_percentage": 100 * after_minus9_counts[0] / len(after_all_minus9),
    "after_all_minus9_class_1_percentage": 100 * after_minus9_counts[1] / len(after_all_minus9),
    "class_0_count": int(final_counts[0]), "class_1_count": int(final_counts[1]),
    "class_0_percentage": 100 * final_counts[0] / len(modelling_df),
    "class_1_percentage": 100 * final_counts[1] / len(modelling_df),
}])
modelling_cohort_audit.to_csv(ARTIFACT_DIR / "modelling_cohort_audit.csv", index=False)
display(modelling_cohort_audit)

,raw_rows,all_minus9_rows_excluded,remaining_rows_after_all_minus9_exclusion,duplicate_excess_excluded,final_modelling_rows,raw_class_0_count,raw_class_1_count,raw_class_0_percentage,raw_class_1_percentage,after_all_minus9_class_0_count,after_all_minus9_class_1_count,after_all_minus9_class_0_percentage,after_all_minus9_class_1_percentage,class_0_count,class_1_count,class_0_percentage,class_1_percentage
0,10459,588,9871,1,9870,5000,5459,47.805718,52.194282,4735,5136,47.968797,52.031203,4735,5135,47.973658,52.026342


## 3. Remaining predictor profiles and frozen split

All final 22-feature predictor profiles are unique, so an ordinary stratified 80/20 split is used. Original row indexes are retained in the frozen partition artifacts.

In [3]:
X = modelling_df[selected_features].copy()
y = modelling_df[TARGET].copy()
profile_sizes = X.groupby(selected_features, dropna=False).size()
unique_profile_count = len(profile_sizes)
duplicate_profile_groups = int(profile_sizes.gt(1).sum())
maximum_group_size = int(profile_sizes.max())

profile_audit = pd.DataFrame([{
    "modelling_rows": len(X), "unique_predictor_profiles": unique_profile_count,
    "duplicate_predictor_profile_groups": duplicate_profile_groups, "maximum_group_size": maximum_group_size,
}])
display(profile_audit)
assert duplicate_profile_groups == 0 and unique_profile_count == len(X)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
)
split_strategy = "train_test_split with stratify=is_at_risk; all remaining predictor profiles unique"

train_indices = pd.DataFrame({"row_index": X_train.index.astype(int)})
test_indices = pd.DataFrame({"row_index": X_test.index.astype(int)})
train_indices.to_csv(ARTIFACT_DIR / "train_indices.csv", index=False)
test_indices.to_csv(ARTIFACT_DIR / "test_indices.csv", index=False)

train_set, test_set = set(X_train.index), set(X_test.index)
excluded_set = set(excluded_all_minus9_indices) | set(excluded_duplicate_indices)
assert train_set.isdisjoint(test_set)
assert train_set | test_set == set(modelling_df.index)
assert excluded_set.isdisjoint(train_set | test_set)

train_profiles = pd.util.hash_pandas_object(X_train, index=False)
test_profiles = pd.util.hash_pandas_object(X_test, index=False)
assert set(train_profiles).isdisjoint(set(test_profiles))

split_summary = pd.DataFrame({
    "partition": ["training", "test"], "rows": [len(X_train), len(X_test)],
    "class_0_count": [int(y_train.eq(0).sum()), int(y_test.eq(0).sum())],
    "class_1_count": [int(y_train.eq(1).sum()), int(y_test.eq(1).sum())],
    "class_1_percentage": [100 * y_train.mean(), 100 * y_test.mean()],
})
display(split_summary)

,modelling_rows,unique_predictor_profiles,duplicate_predictor_profile_groups,maximum_group_size
0,9870,9870,0,1


,partition,rows,class_0_count,class_1_count,class_1_percentage
0,training,7896,3788,4108,52.026342
1,test,1974,947,1027,52.026342


## 4. Training-fitted special-value transformer

For each training-observed feature/code pair, the transformer stores a separate indicator. Original measurements are converted to missing only inside the transformer and median-imputed from training data without scaling.

In [4]:
class HELOCSpecialValuePreprocessor(BaseEstimator, TransformerMixin):
    """Median-impute HELOC measurements and append training-observed special-code indicators."""
    def __init__(self, feature_names, special_codes=(-7, -8, -9)):
        self.feature_names = feature_names
        self.special_codes = special_codes

    def fit(self, X, y=None):
        frame = self._frame(X)
        self.feature_names_in_ = np.asarray(self.feature_names, dtype=object)
        self.indicator_pairs_ = [
            (feature, int(code)) for feature in self.feature_names for code in self.special_codes
            if frame[feature].eq(code).any()
        ]
        numeric = frame[self.feature_names].mask(frame[self.feature_names].isin(self.special_codes))
        self.imputer_ = SimpleImputer(strategy="median", keep_empty_features=True)
        self.imputer_.fit(numeric)
        self.n_features_in_ = len(self.feature_names)
        return self

    def transform(self, X):
        check_is_fitted(self, ["imputer_", "indicator_pairs_"])
        frame = self._frame(X)
        numeric = frame[self.feature_names].mask(frame[self.feature_names].isin(self.special_codes))
        imputed = self.imputer_.transform(numeric)
        indicators = np.column_stack([
            frame[feature].eq(code).astype(np.int8).to_numpy()
            for feature, code in self.indicator_pairs_
        ]) if self.indicator_pairs_ else np.empty((len(frame), 0), dtype=np.int8)
        return np.hstack([imputed, indicators]).astype(float, copy=False)

    def get_feature_names_out(self, input_features=None):
        check_is_fitted(self, ["imputer_", "indicator_pairs_"])
        numeric_names = [f"{feature}__numeric_imputed" for feature in self.feature_names]
        indicator_names = [f"{feature}__is_minus{abs(code)}" for feature, code in self.indicator_pairs_]
        return np.asarray(numeric_names + indicator_names, dtype=object)

    def _frame(self, X):
        if isinstance(X, pd.DataFrame):
            missing = set(self.feature_names) - set(X.columns)
            if missing: raise ValueError(f"Missing required features: {sorted(missing)}")
            return X.loc[:, self.feature_names].copy()
        return pd.DataFrame(X, columns=self.feature_names)

preprocessor = HELOCSpecialValuePreprocessor(selected_features, SPECIAL_CODES)
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

indicator_pairs = preprocessor.indicator_pairs_
print("Underlying numeric columns:", len(selected_features))
print("Special-indicator columns:", len(indicator_pairs))
print("Total transformed features:", X_train_processed.shape[1])
print("Training matrix shape:", X_train_processed.shape)
print("Test matrix shape:", X_test_processed.shape)

Underlying numeric columns: 22
Special-indicator columns: 11
Total transformed features: 33
Training matrix shape: (7896, 33)
Test matrix shape: (1974, 33)


## 5. Training-only leakage and schema audit

Indicator configuration and medians are checked against training data. Test-only special states are reported but cannot create new columns.

In [5]:
training_observed_pairs = [
    (feature, int(code)) for feature in selected_features for code in SPECIAL_CODES
    if X_train[feature].eq(code).any()
]
assert indicator_pairs == training_observed_pairs

training_numeric = X_train.mask(X_train.isin(SPECIAL_CODES))
expected_medians = training_numeric.median().to_numpy(dtype=float)
assert np.allclose(preprocessor.imputer_.statistics_, expected_medians, equal_nan=True)

test_only_pairs = [
    (feature, int(code)) for feature in selected_features for code in SPECIAL_CODES
    if X_test[feature].eq(code).any() and (feature, int(code)) not in training_observed_pairs
]
print("Special codes present only in test:", test_only_pairs if test_only_pairs else "None")
assert X_train_processed.shape[1] == X_test_processed.shape[1]

special_split_rows = []
for feature in selected_features:
    for code_value in SPECIAL_CODES:
        full_count = int(X[feature].eq(code_value).sum())
        training_count = int(X_train[feature].eq(code_value).sum())
        test_count = int(X_test[feature].eq(code_value).sum())
        if full_count:
            special_split_rows.append({"feature": feature, "special_code": int(code_value),
                "full_modelling_count": full_count, "training_count": training_count, "test_count": test_count,
                "present_in_training": training_count > 0, "present_in_test": test_count > 0})
special_value_split_audit = pd.DataFrame(special_split_rows)
special_value_split_audit.to_csv(ARTIFACT_DIR / "special_value_split_audit.csv", index=False)
display(special_value_split_audit)

Special codes present only in test: None


,feature,special_code,full_modelling_count,training_count,test_count,present_in_training,present_in_test
0,months_since_first_trade,-8,239,191,48,True,True
1,months_since_last_illegal_trade,-7,4664,3745,919,True,True
2,months_since_last_illegal_trade,-8,176,132,44,True,True
3,months_since_last_inquiry_not_recent,-7,1855,1497,358,True,True
4,months_since_last_inquiry_not_recent,-8,476,377,99,True,True
5,net_fraction_of_revolving_burden,-8,186,158,28,True,True
6,net_fraction_of_installment_burden,-8,3418,2726,692,True,True
7,nr_revolving_trades_with_balance,-8,156,130,26,True,True
8,nr_installment_trades_with_balance,-8,861,699,162,True,True
9,nr_banks_with_high_ratio,-8,583,467,116,True,True


## 6. Transformed feature mapping

The mapping links every output column to its original feature and representation. It will support later aggregation of model explanations.

In [6]:
display_lookup = semantics_stage02.set_index("raw_feature_name")["display_name"].to_dict()
mapping_rows = []
for feature in selected_features:
    mapping_rows.append({"transformed_feature": f"{feature}__numeric_imputed", "original_feature": feature,
        "display_feature": display_lookup[feature], "representation_type": "numeric_imputed", "special_code": np.nan})
for feature, code_value in indicator_pairs:
    mapping_rows.append({"transformed_feature": f"{feature}__is_minus{abs(code_value)}", "original_feature": feature,
        "display_feature": display_lookup[feature], "representation_type": "special_indicator", "special_code": code_value})
transformed_feature_mapping = pd.DataFrame(mapping_rows)
assert transformed_feature_mapping.transformed_feature.tolist() == preprocessor.get_feature_names_out().tolist()
transformed_feature_mapping.to_csv(ARTIFACT_DIR / "transformed_feature_mapping.csv", index=False)
display(transformed_feature_mapping)

,transformed_feature,original_feature,display_feature,representation_type,special_code
0,months_since_first_trade__numeric_imputed,months_since_first_trade,Months Since First Trade,numeric_imputed,NaN
1,months_since_last_trade__numeric_imputed,months_since_last_trade,Months Since Last Trade,numeric_imputed,NaN
2,average_duration_of_resolution__numeric_imputed,average_duration_of_resolution,Average Duration Of Resolution,numeric_imputed,NaN
3,number_of_satisfactory_trades__numeric_imputed,number_of_satisfactory_trades,Number Of Satisfactory Trades,numeric_imputed,NaN
4,nr_trades_insolvent_for_over_60_days__numeric_...,nr_trades_insolvent_for_over_60_days,Number Of Trades Insolvent For Over 60 Days,numeric_imputed,NaN
5,nr_trades_insolvent_for_over_90_days__numeric_...,nr_trades_insolvent_for_over_90_days,Number Of Trades Insolvent For Over 90 Days,numeric_imputed,NaN
6,percentage_of_legal_trades__numeric_imputed,percentage_of_legal_trades,Percentage Of Legal Trades,numeric_imputed,NaN
7,months_since_last_illegal_trade__numeric_imputed,months_since_last_illegal_trade,Months Since Last Illegal Trade,numeric_imputed,NaN
8,maximum_illegal_trades_over_last_year__numeric...,maximum_illegal_trades_over_last_year,Maximum Illegal Trades Over Last Year,numeric_imputed,NaN
9,maximum_illegal_trades__numeric_imputed,maximum_illegal_trades,Maximum Illegal Trades,numeric_imputed,NaN


## 7. Processed-data validation

Both partitions use the same frozen schema. No transformed values are missing or infinite.

In [7]:
validation = pd.DataFrame([
    {"partition": "training", "rows": X_train_processed.shape[0], "columns": X_train_processed.shape[1],
     "nan_count": int(np.isnan(X_train_processed).sum()), "infinite_count": int(np.isinf(X_train_processed).sum())},
    {"partition": "test", "rows": X_test_processed.shape[0], "columns": X_test_processed.shape[1],
     "nan_count": int(np.isnan(X_test_processed).sum()), "infinite_count": int(np.isinf(X_test_processed).sum())},
])
display(validation)
assert validation.nan_count.sum() == 0 and validation.infinite_count.sum() == 0
assert X_train_processed.shape[1] == len(transformed_feature_mapping) == X_test_processed.shape[1]

,partition,rows,columns,nan_count,infinite_count
0,training,7896,33,0,0
1,test,1974,33,0,0


## 8. Freeze preprocessing and row policy

The saved artifacts reconstruct transformed matrices from the immutable raw data, exclusions, indexes and preprocessor. Redundant processed matrices are not saved.

In [8]:
PREPROCESSOR_PATH = ARTIFACT_DIR / "preprocessor.joblib"
joblib.dump(preprocessor, PREPROCESSOR_PATH)
reloaded_preprocessor = joblib.load(PREPROCESSOR_PATH)
assert np.array_equal(reloaded_preprocessor.transform(X_test), X_test_processed)

def distribution(series):
    counts = series.value_counts().sort_index()
    return {str(int(value)): {"count": int(counts[value]), "percentage": float(100 * counts[value] / len(series))} for value in counts.index}

metadata = {
    "dataset": "mstz/heloc risk configuration", "raw_snapshot_hash": actual_hash,
    "raw_rows": len(raw_df), "excluded_all_minus9_rows": len(excluded_all_minus9_indices),
    "excluded_duplicate_rows": len(excluded_duplicate_indices), "final_modelling_rows": len(modelling_df),
    "selected_features": selected_features,
    "excluded_features": [{"feature": "estimate_of_risk", "reason": "Excluded from the primary model because its construction and decision-time provenance are unresolved, so target/score leakage cannot be ruled out."}],
    "target": TARGET, "target_semantics": {"0": "not at risk", "1": "at risk"},
    "train_rows": len(X_train), "test_rows": len(X_test),
    "train_target_distribution": distribution(y_train), "test_target_distribution": distribution(y_test),
    "split_strategy": split_strategy, "test_size": TEST_SIZE, "random_state": RANDOM_STATE,
    "special_codes": list(SPECIAL_CODES),
    "special_value_strategy": "training-observed separate code indicators; underlying values converted to missing inside preprocessor",
    "imputation_strategy": "SimpleImputer(strategy=median), fitted on training data only",
    "special_indicator_count": len(indicator_pairs), "transformed_feature_count": X_train_processed.shape[1],
    "numeric_scaling": False, "preprocessor_fit_scope": "training_only",
    "test_only_special_feature_code_pairs": [{"feature": feature, "special_code": code} for feature, code in test_only_pairs],
    "validation_design_stage04": {"primary_holdout": 0.20, "model_development": "5-fold stratified CV on training data only", "random_state": 42, "primary_model": "XGBoost binary classifier", "selection": "training-only CV", "final_test": "after model selection"},
    "predictive_model_trained": False, "cross_validation_executed": False, "xai_executed": False,
}
with open(ARTIFACT_DIR / "preprocessing_metadata.json", "w", encoding="utf-8") as handle: json.dump(metadata, handle, indent=2)

row_policy = {
    "raw_rows": len(raw_df),
    "all_minus9_policy": "exclude complete all--9 predictor profiles before splitting",
    "all_minus9_rows": len(excluded_all_minus9_indices),
    "duplicate_policy": "after all--9 exclusion retain first deterministic complete row and exclude duplicate excess copies",
    "duplicate_excess_rows": len(excluded_duplicate_indices), "final_modelling_rows": len(modelling_df),
    "reasoning": "Remove non-informative conflicting repeated profiles and prevent artificial frequency and partition leakage while preserving the immutable raw snapshot.",
}
with open(ARTIFACT_DIR / "row_policy.json", "w", encoding="utf-8") as handle: json.dump(row_policy, handle, indent=2)
print("Saved:", PREPROCESSOR_PATH)

Saved:

 C:\Users\H P E L I T E\Desktop\XAI_CreditStudy\backend\artifacts\heloc\preprocessor.joblib


## 9. Final quality checks

The frozen outputs satisfy the cohort, partition and preprocessing requirements. Model fitting, cross-validation and XAI remain outside this stage.

In [9]:
assert hashlib.sha256(RAW_PATH.read_bytes()).hexdigest() == EXPECTED_HASH
assert len(raw_df) == 10_459 and raw_df.shape[1] == 24
assert "estimate_of_risk" not in selected_features
assert set(excluded_all_minus9_indices).isdisjoint(train_set | test_set)
assert set(excluded_duplicate_indices).isdisjoint(train_set | test_set)
assert train_set.isdisjoint(test_set) and train_set | test_set == set(modelling_df.index)
assert set(train_profiles).isdisjoint(set(test_profiles))
assert np.isnan(X_train_processed).sum() == np.isnan(X_test_processed).sum() == 0
assert np.isinf(X_train_processed).sum() == np.isinf(X_test_processed).sum() == 0
assert len(actionability_stage02) == 23
assert actionability_stage02.preliminary_dice_actionability.value_counts().to_dict() == {"non-actionable": 17, "partially actionable": 5, "unsuitable / unresolved": 1}

print("Raw snapshot hash unchanged: True")
print("Raw rows: 10,459")
print("Final modelling rows:", len(modelling_df))
print("Train/test overlap: 0")
print("Cross-partition duplicate profiles: 0")
print("Preprocessor fit scope: training only")
print("Predictive model trained: False")
print("Cross-validation executed: False")
print("SHAP executed: False")
print("LIME executed: False")
print("DiCE executed: False")

Raw snapshot hash unchanged: True
Raw rows: 10,459
Final modelling rows: 9870
Train/test overlap: 0
Cross-partition duplicate profiles: 0
Preprocessor fit scope: training only
Predictive model trained: False
Cross-validation executed: False
SHAP executed: False
LIME executed: False
DiCE executed: False


## 10. Stage 04 hand-off

Stage 04 may use the frozen holdout and training-only five-fold stratified validation design. The final test partition must remain unused until model selection is complete.